In [9]:
from typing import Optional
from dotenv import load_dotenv
load_dotenv()
import openpyxl
import os
import pandas as pd
import logging as logger
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI, OpenAI
from app.utils.azure_ai_search_retriever import AzureAISearchRetriever, SearchCredential
from app.utils.rag_orchestrator import RAGOrchestrator
from typing import List
import base64
import openai
import weave

logger.getLogger().setLevel(logger.WARN)


In [10]:

def create_search_credential() -> SearchCredential:
    """Creates the appropriate Azure Search credential based on environment variables."""
    api_key = os.getenv("AZURE_SEARCH_API_KEY")
    if api_key:
        logger.info("Using Azure Search API Key credential.")
        return AzureKeyCredential(api_key)


def create_openai_client(purpose: str = "embedding") -> Optional[AzureOpenAI | OpenAI]:
    """Creates an AzureOpenAI client instance based on environment variables."""
    if purpose == "embedding":
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
        api_key = os.getenv("AZURE_OPENAI_API_KEY")
        api_version = os.getenv("AZURE_OPENAI_API_VERSION")
        log_prefix = "Embedding"
    elif purpose == "chat":
        endpoint = None
        api_version = None
        api_key = os.getenv("AZURE_OPENAI_CHAT_API_KEY", os.getenv("AZURE_OPENAI_API_KEY"))
        log_prefix = "Chat"
    else:
        raise ValueError(f"Invalid purpose for OpenAI client: {purpose}")

    try:
        if endpoint is None or api_version is None:
            client = OpenAI(
                api_key=api_key,
            )
        else:
            client = AzureOpenAI(
                azure_endpoint=endpoint,
                api_key=api_key,
                api_version=api_version,
            )
        # Perform a simple test if needed (e.g., list models), but might add latency
        logger.info(
            f"Azure OpenAI client for {log_prefix} initialized successfully (Endpoint: {endpoint}, API Version: {api_version}).")
        return client
    except Exception as e:
        logger.error(f"Failed to initialize Azure OpenAI {log_prefix} client: {e}", exc_info=True)
        return None  # Return None instead of raising, allow graceful failure if only one part is needed

def encode_file(file_path):
    with open(file_path, "rb") as file:
        return base64.b64encode(file.read()).decode('utf-8')


In [11]:
# --- Azure Search Configuration ---
search_service_name = os.getenv("AZURE_SEARCH_SERVICE_NAME")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
search_dns_suffix = os.getenv("AZURE_SEARCH_DNS_SUFFIX", "search.windows.net")

if not search_service_name or not index_name:
    raise ValueError("Missing required environment variables: AZURE_SEARCH_SERVICE_NAME and AZURE_SEARCH_INDEX_NAME")

search_service_endpoint = f"https://{search_service_name}.{search_dns_suffix}"
search_credential = create_search_credential()  # Handles API key, SPN, DefaultAzureCredential

# --- Azure OpenAI Configuration ---
# Embedding Client (Optional for Retriever if only doing text search)
openai_embedding_client = create_openai_client(purpose="embedding")
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Chat Client (Required for Orchestrator)
openai_chat_client = create_openai_client(purpose="chat")
openai_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

if not openai_chat_client or not openai_chat_deployment:
    raise ValueError(
        "Azure OpenAI Chat client configuration (Endpoint, Key, Version, Deployment) is required and incomplete.")

In [12]:
# --- Instantiate Retriever ---
retriever = AzureAISearchRetriever(
    search_service_endpoint=search_service_endpoint,
    index_name=index_name,
    search_credential=search_credential,
    openai_client=openai_embedding_client,  # Pass the client instance
    openai_embedding_deployment=openai_embedding_deployment  # Pass the deployment name
    # select_fields=None, # Use default
    # embedding_dimension=1536 # Use default
)
logger.info("AzureAISearchRetriever instantiated.")

# --- Instantiate Orchestrator ---
orchestrator = RAGOrchestrator(
    retriever=retriever,
    chat_client=openai_chat_client,  # Pass the client instance
    chat_deployment=openai_chat_deployment  # Pass the deployment name
    # system_prompt="Your custom system prompt here", # Optional
    # context_template="Source: {source}\nContent: {content}\n---\n", # Optional
)
logger.info("RAGOrchestrator instantiated.")

In [13]:
# --- Example Query ---
user_query = "Why do we need to do Business Process Re-engineering as a part of implementing an HIS/EHR? Note: Your answer must be in your own words."
logger.info(f"Answering query: '{user_query}'")

answer, sources = orchestrator.answer_query(
    user_query=user_query,
    top_k_retrieval=3,
    search_type="hybrid",  # Try hybrid search
    use_semantic_ranking=False  # Set to True if configured and desired
)

if answer:
    print("\n--- Answer ---")
    print(answer)
    print("\n--- Sources ---")
    if sources:
        for source in sources:
            print(f"- {source}")
    else:
        print("No sources were cited (or retrieval failed).")
else:
    print("\nFailed to get an answer.")

2025-04-01 02:47:19,784 - ERROR - Azure OpenAI API error during embedding generation: 404 - Error code: 404 - {'error': {'code': '404', 'message': 'Resource not found'}}
Traceback (most recent call last):
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\app\utils\azure_ai_search_retriever.py", line 136, in _generate_embeddings
    response = self.openai_client.embeddings.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-packages\openai\resources\embeddings.py", line 128, in create
    return self._post(
           ^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-packages\openai\_base_client.py", line 1242, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MHS\Desktop\PROJECTS\ml-bu-autograder\venv\Lib\site-pack


--- Answer ---
I couldn't find any relevant documents to answer your question.

--- Sources ---
No sources were cited (or retrieval failed).


## Load Google Drive Files
Since our datasets live in Google Drive, we connect to our data source. This particular method assumes you have [Drive for Desktop](https://dl.google.com/drive-file-stream/GoogleDriveSetup.exe) installed on your computer and you are accessing a local path. For this POC, we focus on assignment 2.

In [14]:
class StudentResponse:
    rubric_path: str
    submission_path: str
    def __init__(self, rubric_path, submission_path):
        self.rubric_path = rubric_path
        self.submission_path = submission_path

In [15]:
import os

# Check if the directory exists and list its contents

base_dir = r"G:\My Drive\met_data\Quiz_1\24sprgmetcs581_o1 Quiz 1.xlsx"
if os.path.exists(base_dir):
    print("✅ Path exists!")
    print("Contents:", os.listdir(base_dir))  # List files/folders inside
else:
    print("❌ Path does NOT exist!")


✅ Path exists!


NotADirectoryError: [WinError 267] The directory name is invalid: 'G:\\My Drive\\met_data\\Quiz_1\\24sprgmetcs581_o1 Quiz 1.xlsx'

In [16]:
# Initialize list to hold student responses
student_responses: List[StudentResponse] = []
# Relevant material list
relevant_material = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\CS581 Assignment 2 HIS Clinical (EHR) Functional Requirements 2025 Spring 1.pdf"
assignment_requirements = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\CS581 Assignment 2 HIS Clinical (EHR) Functional Requirements 2025 Spring 1.pdf"

In [17]:
# Define the base directory
base_dir = r"G:\My Drive\met_data\Assignment 2 – EHR Functional Requirements Worksheet\24fallmetcs581_m1 submissions and rubrics"
# Iterate through student directories
for student_number in os.listdir(base_dir):
    student_path = os.path.join(base_dir, student_number)

    if os.path.isdir(student_path):  # Ensure it's a directory
        rubric_path = os.path.join(student_path, "rubric.docx")
        submission_path = os.path.join(student_path, "submission.pptx")

        # Check if both expected files exist
        if os.path.exists(rubric_path) and os.path.exists(submission_path):
            student_responses.append(StudentResponse(rubric_path=rubric_path, submission_path=submission_path))

In [18]:
response = openai_chat_client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "Please analyze the attached file."
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(relevant_material),
                    "file_data": f"data:application/pdf;base64,{encode_file(relevant_material)}"
                }
            ]
        }
    ],
    text={
        "format": {
            "type": "text"
        }
    },
    reasoning={},
    tools=[],
    temperature=1,
    max_output_tokens=2048,
    top_p=1,
    store=True
)


In [19]:
from IPython.display import Markdown, display
display(Markdown(response.output[0].content[0].text))

The document is an assignment for the CS581 course about developing a high-level EHR Functional Requirements document for Virginia Women’s Center’s deployment of an EHR system. Key components include:

1. **Objective**: Craft a functional requirements document focusing on the clinical aspects of an EHR system.

2. **Resources**:
   - VWC case study.
   - CMS Medicare MIPS Quality Payment Program.

3. **Task**:
   - Identify the top 10-12 EHR functions relevant to VWC's needs and CMS PI requirements.
   - Use a Microsoft Excel Functional Requirements spreadsheet to evaluate these functions.

4. **Evaluation Criteria**:
   - Importance ratings (1-4) for each function.
   - Notes explaining the importance rating related to VWC.

5. **Submission Instructions**:
   - Completed spreadsheet submission on the class site.
   - File naming convention: lastname_firstname Assignment 2.

6. **Grading Rubric**:
   - Understanding of EHR functions and their importance.
   - Identification and relevance of functions to VWC and government goals.
   - Clarity, quality, and documentation (including AI usage).

This assignment aims to integrate practical knowledge from the case study and regulatory requirements to define critical EHR functionalities.


Test out with prompt technique to observe the varied results


In [20]:
response = openai_chat_client.responses.create(
    model="gpt-4o",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": """You are an expert document analyst. Analyze the attached file and provide a structured response with:  
                    1. A brief summary (5-7 sentences).  
                    2. The top 3 key insights.  
                    3. Any contradictions, biases, or missing information.  
                    4. Recommendations for improvement (if applicable).  
                    Format your response clearly with headings."""
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": os.path.basename(relevant_material),
                    "file_data": f"data:application/pdf;base64,{encode_file(relevant_material)}"
                }
            ]
        }
    ],
    text={"format": {"type": "text"}},
    reasoning={},
    tools=[],
    temperature=0.5,
    max_output_tokens=1500,
    top_p=0.9,
    store=True
)


In [21]:
from IPython.display import Markdown, display
display(Markdown(response.output[0].content[0].text))

## Summary

The document outlines an assignment for a course on Electronic Health Records (EHR) functional requirements, specifically focusing on the Virginia Women's Center (VWC). The task involves developing a high-level EHR Functional Requirements document, emphasizing clinical functions within a Health Information System (HIS). Students are required to review the VWC case study and the CMS Medicare MIPS Quality Payment Program requirements to identify the top 10-12 functions necessary for VWC's EHR system. The assignment includes filling out a Microsoft Excel spreadsheet with the identified functions, their importance, and notes explaining the ratings. The submission will be graded based on understanding, identification of key functions, their relevance to VWC, and alignment with government goals like MIPS and PI.

## Key Insights

1. **Focus on Clinical Functions**: The assignment emphasizes identifying clinical functions critical to the implementation of an EHR system at VWC, highlighting the importance of aligning these functions with organizational needs and regulatory requirements.

2. **Integration with CMS Requirements**: A significant part of the task is to ensure that the EHR functions meet the CMS Medicare MIPS Quality Payment Program's Promoting Interoperability requirements, underscoring the need for compliance with federal standards.

3. **Structured Evaluation**: The use of a structured Excel spreadsheet for evaluating the importance and fulfillment of each function ensures a systematic approach to assessing the EHR system's capabilities.

## Contradictions, Biases, or Missing Information

- **Missing Context on VWC Needs**: The document does not provide specific details about the Virginia Women's Center's unique needs or challenges, which are crucial for tailoring the EHR functions.
- **Potential Bias in Importance Ratings**: The subjective nature of the importance ratings (1-4 scale) may introduce bias unless clear guidelines are provided for consistent evaluation.

## Recommendations for Improvement

1. **Provide Detailed Context**: Include more information about VWC's specific operational needs and challenges to guide students in selecting relevant EHR functions.

2. **Clarify Rating Criteria**: Offer detailed criteria or examples for the importance ratings to minimize bias and ensure consistency across different submissions.

3. **Include Vendor Evaluation Guidance**: Although vendor scoring is not required for this assignment, providing a brief overview or future guidance on evaluating vendor capabilities could enhance students' understanding of comprehensive EHR system assessment.

In [22]:
excel = pd.read_excel(r'G:\My Drive\met_data\Quiz_1\24fallmetcs581_m1 Quiz 1.xlsx', sheet_name=None)
for sheet_name, df_sheet in excel.items():
    var_name = f"df_{sheet_name.replace(' ', '_')}"
    globals()[var_name] = df_sheet
    print(f"Assigned DataFrame to: {var_name}")


Assigned DataFrame to: df_Question_Details
Assigned DataFrame to: df_Student_Submissions


In [23]:
df_Student_Quiz_Responses = pd.read_excel(r'G:\My Drive\met_data\Quiz_1\24fallmetcs581_m1 Quiz 1.xlsx', sheet_name='Student Submissions')

In [24]:

pd.set_option('display.max_rows', 5)  # Limits rows displayed
pd.set_option('display.max_columns', 10)  # Limits columns displayed
df_Student_Quiz_Responses.columns

Index(['Unnamed: 0', 'question 13 answer', 'question 13 score',
       'question 13 feedback', 'question 14 answer', 'question 14 score',
       'question 14 feedback', 'additional feedback '],
      dtype='object')

In [ ]:
def grade_student_response(course_material_path, student_response):
    # Encode the course material
    encoded_material = encode_file(course_material_path)
    
    # Create the grading prompt
    grading_prompt = f"""
    You are an expert grading assistant. Your task is to evaluate a student's response against the provided course material.
    
    Course Material Content (from attached file):
    [The system will have access to the attached course material]
    
    Student Response:
    {student_response}
    
   "As a health informatics professional, I will assess each student's response to the assignment based on the lecture material provided. The evaluation will include:

A score out of 20 reflecting the quality and completeness of the response.

Detailed feedback for each criterion, including an analysis of how well the response aligns with the concepts discussed in the lecture.

Specific suggestions for improvement to enhance the quality and depth of understanding.

Relevant quotes from the course material that support my evaluation and recommendations.

This grading process ensures that students receive constructive feedback tailored to the health informatics field, guiding them toward a deeper understanding of the subject matter.
    """
    
    response = openai_chat_client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": grading_prompt
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": os.path.basename(course_material_path),
                        "file_data": f"data:application/pdf;base64,{encoded_material}"
                    }
                ]
            }
        ],
        text={"format": {"type": "text"}},
        reasoning={},
        tools=[],
        temperature=0.3,  # Lower temperature for more consistent grading
        max_output_tokens=2000,
        top_p=0.9,
        store=True
    )
    return (response.output[0].content[0].text)


In [26]:
COURSE_MATERIAL = r"G:\My Drive\met_data\Quiz_1\Mod 1 BPR & Workflow - Lecture Slides.pdf"
# Grade each answer in the DataFrame
df_Student_Quiz_Responses["question 13 grade"] = df_Student_Quiz_Responses["question 13 answer"].apply(
    lambda answer: grade_student_response(COURSE_MATERIAL, answer)
)


In [28]:
df_Student_Quiz_Responses

,Unnamed: 0,question 13 answer,question 13 score,question 13 feedback,question 14 answer,question 14 score,question 14 feedback,additional feedback,question 13 grade,question 14 grade
0,student 1,since BPR shows and indicates the currents dat...,12 / 14,Your response covers several important points ...,Enterprise Architecture increase and enhance i...,14 / 14,Nice work!,Feedback on Short Answer Questions:\n\nQuestio...,**Score: 85/100**\n\n**Detailed Feedback:**\n\...,**Score: 85/100**\n\n**Feedback:**\n\n1. **Acc...
1,student 2,When we look at the benefits from an IT operat...,14/14,Great work!,Business Process Re-engineering is very import...,14/14,Great work!,Question 13: Great work! \n\nQuestion 14: Grea...,**Evaluation of Student Response**\n\n**Score:...,**Score: 75/100**\n\n**Feedback:**\n\n1. **Acc...
...,...,...,...,...,...,...,...,...,...,...
20,student 21,\t\nOne significant benefit of using an Enterp...,13/14,"Overall, good work! Elaborating on how enhance...",Optimization of Workflows: BPR allows organiza...,14/14,Great job!,"Question 13: Overall, good work! Elaborating o...",**Score: 85/100**\n\n**Feedback:**\n\n1. **Acc...,**Evaluation of Student Response**\n\n**Score:...
21,student 22,Optimized use of resources and efforts.\n\nHav...,2025-11-14 00:00:00,You effectively identified a relevant benefit ...,\t\nI think that Business Process Re-engineeri...,2025-11-14 00:00:00,You’ve effectively highlighted the importance ...,Question 13: You effectively identified a rele...,**Score: 85/100**\n\n**Feedback:**\n\n1. **Acc...,**Score: 70/100**\n\n**Feedback:**\n\n1. **Acc...


In [27]:
COURSE_MATERIAL = r"G:\My Drive\met_data\Quiz_1\Mod 1 BPR & Workflow - Lecture Slides.pdf"

# Grade each answer in the DataFrame
df_Student_Quiz_Responses["question 14 grade"] = df_Student_Quiz_Responses["question 14 answer"].apply(
    lambda answer: grade_student_response(COURSE_MATERIAL, answer)
)
